# ragfoundry on Google Colab

This notebook installs [ragfoundry](https://pypi.org/project/ragfoundry/) from PyPI and runs a complete RAG pipeline: ingest documents, ask questions, stream an answer, chat with follow-up questions, limit the context window, turn on hybrid search and reranking, evaluate, and load the pipeline from YAML.

**Before you start:** add an API key for one LLM provider in Colab's **Secrets** panel (the 🔑 key icon in the left sidebar), and switch on *Notebook access* for it.

| Provider | Secret name | Free key |
|---|---|---|
| Groq | `GROQ_API_KEY` | https://console.groq.com/keys |
| Gemini | `GOOGLE_API_KEY` | https://aistudio.google.com/apikey |
| OpenAI | `OPENAI_API_KEY` | (paid) |
| Claude | `ANTHROPIC_API_KEY` | (paid) |

Embeddings run locally with Hugging Face, so they need no key. A CPU runtime is enough.

## 1. Install

In [ ]:
%pip install -q "ragfoundry[huggingface,chroma,groq,gemini,openai,claude,pdf,docx,web]"

In [ ]:
import ragfoundry
print("ragfoundry", ragfoundry.__version__)

## 2. Choose the LLM

Set `PROVIDER` to the provider whose key you added. Model names change often, so if one is rejected, replace it with a current model from that provider's docs.

In [ ]:
from google.colab import userdata

PROVIDER = "groq"  # groq | gemini | openai | claude

MODELS = {
    "groq":   ("llama-3.3-70b-versatile", "GROQ_API_KEY"),
    "gemini": ("gemini-2.5-flash",        "GOOGLE_API_KEY"),
    "openai": ("gpt-4o-mini",             "OPENAI_API_KEY"),
    "claude": ("claude-haiku-5-5",        "ANTHROPIC_API_KEY"),
}
MODEL, SECRET_NAME = MODELS[PROVIDER]
API_KEY = userdata.get(SECRET_NAME)  # raises an error if the secret is missing or not shared

LLM = {"provider": PROVIDER, "model": MODEL, "api_key": API_KEY, "temperature": 0}
print(f"Using {PROVIDER} / {MODEL}")

## 3. Create sample documents

Three short files so you can check the answers yourself. Section 9 shows how to use your own files.

In [ ]:
from pathlib import Path

docs = Path("docs")
docs.mkdir(exist_ok=True)

(docs / "solar.md").write_text("""# Solar power
Silicon solar cells convert about 20 percent of sunlight into electricity.
Perovskite cells are a newer technology that has passed 25 percent efficiency in the lab,
but they degrade faster than silicon outdoors. Panels produce direct current, so an
inverter converts it to alternating current for the home.
""")

(docs / "sourdough.md").write_text("""# Sourdough baking
A sourdough starter is a culture of wild yeast and lactic acid bacteria.
It needs regular feeding with flour and water, usually once a day at room temperature.
Bulk fermentation typically takes four to six hours at 24 degrees Celsius.
Bakers bake the loaf in a preheated Dutch oven at about 250 degrees Celsius.
""")

(docs / "coffee.txt").write_text("""Coffee brewing guide.
The recommended water temperature for brewing coffee is 90 to 96 degrees Celsius.
A common ratio is 1 gram of coffee to 16 grams of water.
Espresso is brewed at around 9 bars of pressure in 25 to 30 seconds.
""")

print(sorted(p.name for p in docs.iterdir()))

## 4. Build the pipeline and ingest

In [ ]:
from ragfoundry import RAGConfig, RAGPipeline

config = RAGConfig.model_validate({
    "persist_dir": "rag_basic",
    "embeddings": {"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2"},
    "vector_store": {"provider": "chroma", "collection_name": "demo"},
    "llm": LLM,
    "chunking": {"strategy": "recursive", "chunk_size": 300, "chunk_overlap": 50},
    "retrieval": {"top_k": 3},
})

rag = RAGPipeline(config)
report = rag.ingest("docs")
print(report.to_dict())

Run the ingest cell again: `chunks_added` drops to 0 and `chunks_unchanged` goes up, because ingestion only processes chunks that changed.

In [ ]:
print(rag.ingest("docs").to_dict())
print(rag.sources())

## 5. Ask questions

In [ ]:
result = rag.query("How efficient are silicon solar cells?")
print(result.answer)
print("Sources:", result.sources)

In [ ]:
# A question the documents can't answer: the pipeline should say so.
result = rag.query("Who won the 2022 football World Cup?")
print(result.answer)
print("Abstained:", result.abstained)

In [ ]:
# Retrieval only, without calling the LLM.
for doc in rag.retrieve("What temperature should coffee be brewed at?"):
    print(doc.metadata["source"], "|", doc.page_content[:80].replace("\n", " "))

In [ ]:
# Only search one file, using a metadata filter. Sources are stored as absolute paths.
sourdough = str(Path("docs/sourdough.md").resolve())
result = rag.query("What temperature is used?", where={"source": sourdough})
print(result.answer, result.sources)

## 6. Stream an answer

In [ ]:
for token in rag.stream("How do you take care of a sourdough starter?"):
    print(token, end="", flush=True)

## 6b. Chat with follow-up questions

`rag.chat()` returns a session that remembers the conversation. Follow-up questions such as "how often?" are rewritten into standalone questions before searching, and the last few turns are sent to the LLM.

In [ ]:
chat = rag.chat()
print(chat.ask("What does a sourdough starter need?").answer)

follow_up = chat.ask("How often?")
print("Searched for:", follow_up.standalone_question)
print(follow_up.answer)

saved = chat.to_dicts()        # store this (e.g. as JSON) to resume later
resumed = rag.chat(saved)
print(len(resumed), "turns restored")

## 6c. Context window limits

`context_window` caps the tokens of retrieved passages and of chat history sent with each question. The passage that crosses the budget is shortened rather than dropped.

In [ ]:
small = RAGPipeline(RAGConfig.model_validate({
    "persist_dir": "rag_small",
    "embeddings": {"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2"},
    "vector_store": {"provider": "chroma", "collection_name": "demo_small"},
    "llm": LLM,
    "retrieval": {"top_k": 4},
    "context_window": {
        "max_context_tokens": 80,     # retrieved passages
        "max_history_turns": 3,       # chat turns
        "max_history_tokens": 500,
    },
}))
small.ingest("docs")
result = small.query("Tell me about solar cells and coffee")
print("Context tokens:", result.context_tokens)
for doc in result.documents:
    print(doc.metadata["source"], "| truncated:", doc.metadata.get("truncated", False))

## 7. Advanced retrieval

Hybrid search (BM25 + vectors), a cross-encoder reranker, multi-query rewriting, and the corrective loop that grades retrieved chunks and retries with a rewritten query.

In [ ]:
advanced = RAGPipeline(RAGConfig.model_validate({
    "persist_dir": "rag_advanced",
    "embeddings": {"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2"},
    "vector_store": {"provider": "chroma", "collection_name": "demo_advanced"},
    "llm": LLM,
    "chunking": {"strategy": "recursive", "chunk_size": 300, "chunk_overlap": 50},
    "retrieval": {
        "top_k": 3,
        "fetch_k": 10,
        "hybrid": {"enabled": True},
        "rerank": {"provider": "huggingface", "model": "cross-encoder/ms-marco-MiniLM-L-6-v2"},
        "query_transform": {"strategies": ["multi_query"], "num_queries": 3},
        "corrective": {"enabled": True, "max_retries": 1},
    },
}))
advanced.ingest("docs")

result = advanced.query("What pressure is espresso made at?")
print(result.answer)
print("Queries tried:", result.queries)
print("Sources:", result.sources, "| retries:", result.retries)

## 8. Evaluate

Retrieval metrics (hit rate, precision, recall, MRR, MAP, nDCG) need no LLM. Generation metrics (faithfulness, answer relevance, correctness) use the LLM as a judge, so they make extra API calls.

In [ ]:
from ragfoundry import EvalSample

samples = [
    EvalSample(question="How efficient are silicon solar cells?",
               reference_answer="About 20 percent.", relevant_sources=["solar.md"]),
    EvalSample(question="What does a sourdough starter need?",
               reference_answer="Regular feeding with flour and water.", relevant_sources=["sourdough.md"]),
    EvalSample(question="What water temperature is best for coffee?",
               reference_answer="90 to 96 degrees Celsius.", relevant_sources=["coffee.txt"]),
]

retrieval_report = rag.evaluate(samples, mode="retrieval")
for metric, stats in retrieval_report.summary.items():
    print(f"{metric:<20} {stats['mean']:.3f}")

In [ ]:
full_report = rag.evaluate(samples, mode="all")
for metric, stats in full_report.summary.items():
    print(f"{metric:<20} {stats['mean']:.3f}")

full_report.save("eval_report.json")  # per-question details

Optional RAGAS scoring (installs the `ragas` extra and calls the LLM many times):

```python
%pip install -q "ragfoundry[ragas]"
ragas_report = rag.evaluate(samples, mode="generation", backend="ragas")
print(ragas_report.summary)
```

## 9. Use your own files

Upload PDF, DOCX, TXT, MD, CSV or Excel files, or ingest a web page.

In [ ]:
from google.colab import files

my_docs = Path("my_docs")
my_docs.mkdir(exist_ok=True)
for name, data in files.upload().items():
    (my_docs / name).write_bytes(data)

print(rag.ingest("my_docs").to_dict())

In [ ]:
# Web pages work too (max_depth 0 means only this page).
print(rag.ingest("https://en.wikipedia.org/wiki/Retrieval-augmented_generation").to_dict())
print(rag.query("What is retrieval-augmented generation?").answer)

## 10. YAML config

The same pipeline from a config file. The key is read from an environment variable, so it isn't written into the file.

In [ ]:
import os
os.environ["LLM_API_KEY"] = API_KEY

Path("rag.yaml").write_text(f'''
persist_dir: rag_cli
embeddings:
  provider: huggingface
  model: sentence-transformers/all-MiniLM-L6-v2
vector_store:
  provider: chroma
  collection_name: cli_demo
llm:
  provider: {PROVIDER}
  model: {MODEL}
  api_key: ${{LLM_API_KEY}}
chunking:
  chunk_size: 300
  chunk_overlap: 50
retrieval:
  top_k: 3
  hybrid:
    enabled: true
''')

In [ ]:
yaml_rag = RAGPipeline.from_yaml("rag.yaml")
yaml_rag.ingest("docs")
print(yaml_rag.sources())
print(yaml_rag.query("What ratio of coffee to water is common?").answer)

## 11. Clean up

In [ ]:
for pipeline in (rag, small, advanced, yaml_rag):
    pipeline.close()